# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ b047f042-d811-4966-a5c3-14a50b… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ c80fac98-0e96-4e31-bb39-9fab5e… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-07 ┆ claude-hai ┆ 558.0      ┆ 1124.0    ┆ 16551786  ┆ 8257737   ┆ 0.050966  ┆ 46.272377 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## Kiểm tra bổ sung Gold (do học viên thêm)
Notebook gốc chỉ assert `n_dates >= 7`. Cell dưới kiểm tra đủ hợp đồng Gold theo RUBRIC: Bronze/Silver/Gold có trên đĩa, Silver < Bronze, mỗi model có ≥ 7 ngày, p50 ≤ p95, `cost_usd > 0`, `error_rate ∈ [0, 1]`.

In [6]:
from pathlib import Path
bronze_n = DeltaTable(BRONZE).count()
silver_n = DeltaTable(SILVER).count()
for name, t in [("bronze", BRONZE), ("silver", SILVER), ("gold", GOLD)]:
    print(f"{name:<7} _delta_log present: {(Path(t) / '_delta_log').is_dir()}  ->  {t}")
per_model = gold_df.group_by("model").agg(pl.col("date").n_unique().alias("n_dates")).sort("model")
print(per_model)
checks = {
    "3 tables on disk":         all((Path(t) / "_delta_log").is_dir() for t in (BRONZE, SILVER, GOLD)),
    "silver < bronze":          silver_n < bronze_n,
    "3 models":                 gold_df["model"].n_unique() == 3,
    ">= 7 dates per model":     per_model["n_dates"].min() >= 7,
    "p50 <= p95 (all rows)":    (gold_df["p50_latency_ms"] <= gold_df["p95_latency_ms"]).all(),
    "cost_usd > 0 (all rows)":  (gold_df["cost_usd"] > 0).all(),
    "error_rate in [0,1]":      gold_df["error_rate"].is_between(0, 1).all(),
}
print()
print(f"bronze={bronze_n:,}  silver={silver_n:,}  dropped={bronze_n - silver_n:,}")
print(f"error_rate range: [{gold_df['error_rate'].min():.4f}, {gold_df['error_rate'].max():.4f}]  "
      f"cost_usd range: [{gold_df['cost_usd'].min():.2f}, {gold_df['cost_usd'].max():.2f}]")
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values())

bronze  _delta_log present: True  ->  C:\Users\VUHUY\Desktop\labvinuni20k\K4-Track02-Day18-NguyenVuHuy-2A202602662-Lakehouse-Lab\_lakehouse\bronze\llm_calls_raw
silver  _delta_log present: True  ->  C:\Users\VUHUY\Desktop\labvinuni20k\K4-Track02-Day18-NguyenVuHuy-2A202602662-Lakehouse-Lab\_lakehouse\silver\llm_calls
gold    _delta_log present: True  ->  C:\Users\VUHUY\Desktop\labvinuni20k\K4-Track02-Day18-NguyenVuHuy-2A202602662-Lakehouse-Lab\_lakehouse\gold\llm_daily_metrics
shape: (3, 2)
┌───────────────────┬─────────┐
│ model             ┆ n_dates │
│ ---               ┆ ---     │
│ str               ┆ u32     │
╞═══════════════════╪═════════╡
│ claude-haiku-4-5  ┆ 8       │
│ claude-opus-4-7   ┆ 8       │
│ claude-sonnet-4-6 ┆ 8       │
└───────────────────┴─────────┘

bronze=200,000  silver=190,052  dropped=9,948
error_rate range: [0.0422, 0.0618]  cost_usd range: [13.50, 348.74]
  [PASS] 3 tables on disk
  [PASS] silver < bronze
  [PASS] 3 models
  [PASS] >= 7 dates per model
  [

## Giải thích kết quả (học viên)
| Tiêu chí | Kết quả đo | Ngưỡng |
|---|---|---|
| Bronze / Silver / Gold trên đĩa | cả 3 thư mục có `_delta_log/` (`_lakehouse/{bronze,silver,gold}/…`) | đủ 3 |
| Silver dedup | Bronze **200 000** → Silver **190 052** (loại 9 948 bản trùng `request_id`) | Silver < Bronze |
| Gold | **24** dòng = **8 ngày × 3 model**; p50 ≤ p95 mọi dòng; `cost_usd` ∈ [13.50, 348.74]; `error_rate` ∈ [0.042, 0.062] | ≥ 7 ngày × 3 model |

**Cơ chế.** Bronze giữ JSON thô đúng như log gửi tới (kể cả bản trùng do retry của client) để luôn
có thể replay. Silver parse `raw_json` thành cột có kiểu và **dedup theo `request_id`**: generator
cố ý gieo 9 948 bản trùng, và đúng 9 948 dòng biến mất — nếu không dedup thì cost và error_rate ở Gold
bị đếm hai lần cho các request retry. Gold tổng hợp theo `(date, model)`: p50/p95 latency bằng
`QUANTILE_CONT`, `error_rate = AVG(status <> 'ok')`, `cost_usd` = tokens × giá minh họa của lab.
Ví dụ 2026-04-06: Opus p95 6 013 ms so với Haiku 1 138 ms, nhưng Sonnet tốn nhất (343 USD) vì lưu lượng
token lớn gấp ~6 lần Opus — đúng loại câu hỏi dashboard cost/latency cần trả lời. Gold được partition
theo `date` và Z-order theo `model` cho truy vấn dashboard "lọc theo model".

Có 8 ngày vì dữ liệu 7 ngày UTC tính từ 2026-04-01 tràn sang một phần ngày 2026-04-08 (ngày này
có lượng token ít hơn hẳn — partial day). Notebook gốc chỉ assert `n_dates >= 7`, nên phía trên tôi
thêm cell kiểm tra đủ các điều kiện còn lại của rubric (tất cả PASS).
